# ML-09 — Validation and Research Claim Audit

## 1. Two paper findings + my methodology questions

**Finding 1:** "Updating content leads to an average 40% increase in traffic."
*   **My Methodology Question:** How was causality established here? Since we are looking at observational data, how do we know the traffic increase wasn't due to seasonal trends or a broader site-wide algorithm update happening at the same time as the content update?

**Finding 2:** "Pages with over 2000 words rank higher than shorter pages."
*   **My Methodology Question:** Does this finding account for the confounding variable of backlink profile or domain authority? Highly authoritative sites might simply write longer content, and it is the authority—not the word count itself—driving the rank.

## 2. My model under an honest split (Before/After)

*Here we re-run the Week-5 Random Forest. But this time, instead of randomly splitting the rows, we group the split by `client_id`. This prevents the model from peeking at a client's patterns in the training set and artificially inflating the score on that same client in the test set.*

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score

# 1. Load Data & Define Target/Features (Same as Week 5)
df = pd.read_csv('https://raw.githubusercontent.com/flyrank-bih/flyrank-ml-internship-starter/main/data/raw/content_refresh_anonymized.csv')
df['target_decay'] = np.where((df['clicks_last_30d'] < (df['clicks_prev_30d'] * 0.8)) & (df['impressions_prev_30d'] > 100), 1, 0)
features = ['content_age_days', 'word_count', 'search_volume', 'competition', 'cpc', 'impressions_prev_30d', 'clicks_prev_30d', 'avg_position']
X = df[features].fillna(0)
y = df['target_decay']
groups = df['client_id'] # The grouping variable

# --- BEFORE: Random Split (Week 5) ---
X_train_rand, X_test_rand, y_train_rand, y_test_rand = train_test_split(X, y, test_size=0.2, random_state=42)
rf_rand = RandomForestClassifier(n_estimators=100, random_state=42, max_depth=10)
rf_rand.fit(X_train_rand, y_train_rand)
score_rand = precision_score(y_test_rand, rf_rand.predict(X_test_rand))
print(f"BEFORE (Random Split Precision): {score_rand:.3f}")

# --- AFTER: Honest Grouped Split (Week 6) ---
# We ensure no client in the test set was seen in the training set
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups))

X_train_grp, y_train_grp = X.iloc[train_idx], y.iloc[train_idx]
X_test_grp, y_test_grp = X.iloc[test_idx], y.iloc[test_idx]

rf_grp = RandomForestClassifier(n_estimators=100, random_state=42, max_depth=10)
rf_grp.fit(X_train_grp, y_train_grp)
score_grp = precision_score(y_test_grp, rf_grp.predict(X_test_grp))
print(f"AFTER (Honest Grouped Precision): {score_grp:.3f}")
print("\nObservation: The precision slightly drops when using the honest grouped split because the model can no longer memorize client-specific quirks. This is the true, generalizable performance.")


BEFORE (Random Split Precision): 0.657
AFTER (Honest Grouped Precision): 0.590

Observation: The precision slightly drops when using the honest grouped split because the model can no longer memorize client-specific quirks. This is the true, generalizable performance.


## 3. Leakage audit

*Did any future information sneak into our features?*

**Audit Results:** Our feature set (`content_age_days`, `impressions_prev_30d`, `clicks_prev_30d`, etc.) is strictly historical. The target is defined using `clicks_last_30d`. Because we are not including any `last_30d` columns in our feature array `X`, we have successfully avoided direct label leakage. A potential subtle leak could occur if `avg_position` is calculated over the *entire* 90-day window rather than strictly the historical 60-day window, but based on the schema, it is safe to proceed directionally.

## 4. Claim rewrite

**Original Claim (Too bold):** "Our Random Forest model perfectly identifies decaying content and guarantees that rewriting these pages will recover lost search traffic."

**Rewritten Claim (Honest & Safe):** "Our model **observed** historical patterns to flag pages experiencing severe traffic decay. These predictions provide **directional decision-support** to help prioritize editorial resources, though we cannot guarantee traffic recovery due to external algorithmic factors."

## 5. Self-check

Completed and verified.